In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import hamiltonian_generator

from qiskit.primitives import StatevectorEstimator
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2

from itertools import product

import warnings
warnings.filterwarnings("ignore")

In [2]:
def save_reference_energy(n, h) :
    ansatz = efficient_su2(num_qubits=n, reps=2)
    hamiltonian = hamiltonian_generator.get_ising_hamiltonian(n, h)
    estimator = StatevectorEstimator()
    params = np.load(f'params/params_{n}_{h}.npy')
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )
    np.save(f'data/ref_{n}_{h}', energy)
    return energy

In [3]:
num_sites = [12, 8, 5, 3]
couplings = [0, 0.5, 2, 5]

for n, h in product(num_sites, couplings) :
    energy = save_reference_energy(n, h)
    print(f"n={n}, h={h} \t\t {energy}")

n=12, h=0 		 [-9.77289069]
n=12, h=0.5 		 [-10.42133767]
n=12, h=2 		 [-21.12360144]
n=12, h=5 		 [-60.15831953]
n=8, h=0 		 [-5.74814522]
n=8, h=0.5 		 [-5.05853821]
n=8, h=2 		 [-12.08884312]
n=8, h=5 		 [-30.17605866]
n=5, h=0 		 [-3.85321656]
n=5, h=0.5 		 [-4.28216438]
n=5, h=2 		 [-9.97977128]
n=5, h=5 		 [-25.15255825]
n=3, h=0 		 [-1.9380177]
n=3, h=0.5 		 [-2.29145444]
n=3, h=2 		 [-6.11642846]
n=3, h=5 		 [-15.05072541]


In [4]:
num_sites = [12]
couplings = [0.5, 5]

final_time = 0.25
num_timesteps = 5

In [5]:
estimator = StatevectorEstimator()

for (n, h) in product(num_sites, couplings) :
    dummy = QuantumCircuit(n)
    H = hamiltonian_generator.get_ising_hamiltonian(n, h)
    sim_energy = (
        estimator.run([(dummy, H)]).result()[0].data.evs
    )
    print(f"n={n}, h={h}: \t\t {sim_energy}")
    np.save(f'data/evo_ref_{n}_{h}', sim_energy)

n=12, h=0.5: 		 11.0
n=12, h=5: 		 11.0
